# ENVIRONMENT SETUP
## Detekcja Środowiska Wykonawczego

In [17]:
def is_execution_inside_colab():
    try:
        import google.colab
        ! pip install tonic snntorch
        return True
    except:
        return False

## Konfiguracja Specyficzna dla środowiska

In [18]:
if is_execution_inside_colab():
    datasets_location = "./datasets"
else:
    datasets_location = "../../../datasets"

## Ogólny Setup Środowiska

In [19]:
import os
# Musi byc ustawione PRZED pierwszym uzyciem CUDA - ogranicza fragmentacje alokatora.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import numpy as np
import torch
import tonic
import snntorch
import snntorch.utils as snntorch_utils
import snntorch.functional as snntorch_functional
import matplotlib.pyplot as plt
from torch.utils.checkpoint import checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True


device: cuda


# PARAMETER CONFIG

In [20]:
CONFIG = {
    "data_root": datasets_location,

    # PAMIEC: koszt aktywacji rosnie jak H*W, wiec 128 -> 64 to 4x mniej pamieci.
    # Dla uczciwego benchmarku ta sama wartosc musi byc uzyta w modelu CNN.
    "image_size": 64,

    # PAMIEC: mniejszy batch + akumulacja gradientu = ten sam efektywny batch (16),
    # ale polowa szczytowego zuzycia pamieci.
    "batch_size": 1024,
    "accum_steps": 2,

    "num_workers": 2,
    "n_time_bins": 10,          # pamiec aktywacji rosnie liniowo z T

    # 100 klas obiektow - wspolne dla N-Caltech101 (bez "Faces") i Caltech101
    # po stronie CNN (bez "Faces"). Klasa tla ("BACKGROUND_Google") jest
    # wykluczona z obu pipeline'ow, bo torchvision juz jej nie traktuje jako
    # realnej klasy, a odtwarzanie jej parytetu miedzy sensorami nie daje
    # dodatkowej wartosci porownawczej.
    "num_classes": 100,
    # Jedyne miejsce definicji wykluczonych klas - uzywane zarowno przy
    # filtrowaniu samego datasetu (ponizej), jak i przy budowie class_to_idx,
    # zeby obie liczby nigdy sie nie rozjechaly.
    "excluded_classes": ["BACKGROUND_Google"],
    "ignore_index": 255,

    "lr": 2e-3,
    "epochs": 5,
    "max_steps_per_epoch": None,
    "validation_split_size": 0.2,

    # PAMIEC: gradient checkpointing po krokach czasowych.
    # Zamienia ok. 30% dodatkowego czasu obliczen na duzo mniejsze zuzycie pamieci.
    # Wylacz (False), jesli mierzysz "czysty" czas treningu do benchmarku.
    "use_checkpoint": True,

    # Mixed precision (fp16) - polowa pamieci aktywacji.
    "use_amp": True,
}


# Data Augmentation

In [21]:
import shutil

CACHE_PATH = './cache/ncaltech101'

# Cache tonica jest kluczowany sciezka, NIE transformacja. Po zmianie
# image_size / n_time_bins stare pliki .hdf5 trzeba skasowac, inaczej
# dostaniesz ciche niezgodnosci ksztaltow.
REBUILD_CACHE = True
if REBUILD_CACHE:
    shutil.rmtree(CACHE_PATH, ignore_errors=True)

# DiskCachedDataset tworzy ten katalog w __init__, ale jesli zostal usuniety
# pozniej (albo runtime sie zrestartowal), zapis do cache wywala sie na
# FileNotFoundError. Tworzymy go jawnie.
os.makedirs(CACHE_PATH, exist_ok=True)

sensor_size = (240, 180, 2)
frame_transform = tonic.transforms.Compose([
    tonic.transforms.Downsample(sensor_size=sensor_size, target_size=CONFIG["image_size"]),
    tonic.transforms.ToFrame(
        sensor_size=(CONFIG["image_size"], CONFIG["image_size"], 2),
        n_time_bins=CONFIG["n_time_bins"],
    ),
    lambda x: torch.from_numpy(x).float()
])

dataset = tonic.datasets.NCALTECH101(save_to='./data', transform=frame_transform)

# WAZNE: samo przefiltrowanie listy nazw klas (dalej, przy budowie class_to_idx)
# NIE usuwa probek z datasetu - tonic.datasets.NCALTECH101 laduje kazdy plik .bin
# z kazdego folderu, background wlacznie. Trzeba przefiltrowac same probki,
# analogicznie do FilteredCaltech101 po stronie CNN.
class FilteredEventDataset(torch.utils.data.Dataset):
    """N-Caltech101 bez podanych klas (np. tla). Zachowuje oryginalne etykiety
    tekstowe - remapping na indeksy dzieje sie pozniej, w decode_targets."""

    def __init__(self, base, excluded_classes):
        self.base = base
        excluded = set(excluded_classes)
        self.keep_indices = [i for i, t in enumerate(base.targets) if t not in excluded]

    def __len__(self):
        return len(self.keep_indices)

    def __getitem__(self, idx):
        return self.base[self.keep_indices[idx]]


print("probek przed filtrowaniem:", len(dataset))
dataset = FilteredEventDataset(dataset, CONFIG["excluded_classes"])
print("probek po usunieciu", CONFIG["excluded_classes"], ":", len(dataset))

cached_dataset = tonic.DiskCachedDataset(dataset, cache_path=CACHE_PATH)

# UWAGA: usunieto `all_loader` - byl tworzony i nigdy nie uzywany.

train_size = int((1 - CONFIG["validation_split_size"]) * len(cached_dataset))
test_size = len(cached_dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(cached_dataset, [train_size, test_size])

loader_kwargs = dict(
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    persistent_workers=CONFIG["num_workers"] > 0,
)

train_loader = torch.utils.data.DataLoader(train_dataset, shuffle=True, **loader_kwargs)
test_loader = torch.utils.data.DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print("train:", len(train_dataset), "test:", len(test_dataset))


probek przed filtrowaniem: 8709
probek po usunieciu ['BACKGROUND_Google'] : 8242
train: 6593 test: 1649


In [22]:
from tqdm.auto import tqdm

warmup_loader = torch.utils.data.DataLoader(
    cached_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0
)

for _ in tqdm(warmup_loader, desc="budowanie cache"):
    pass

del warmup_loader
print("cache gotowy:", len(os.listdir(CACHE_PATH)), "plikow")


budowanie cache:   0%|          | 0/9 [00:00<?, ?it/s]

cache gotowy: 8242 plikow


# Basic Block

In [23]:
# Surrogate gradient dla propagacji przez nierozniczkowalne spike'i
spike_grad = snntorch.surrogate.atan()


class StatelessLIF(torch.nn.Module):
    """
    Cienka nakladka na snntorch.Leaky z init_hidden=False.

    POWOD: przy init_hidden=True potencjal blonowy jest trzymany jako atrybut
    modulu. Gradient checkpointing uruchamia forward PONOWNIE podczas backward,
    wiec taki ukryty stan zostalby zaktualizowany drugi raz i gradienty bylyby
    bledne. Tutaj stan jest jawnie przekazywany i zwracany, dzieki czemu
    checkpointing jest poprawny.
    """
    def __init__(self, beta, spike_grad):
        super().__init__()
        self.lif = snntorch.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=False)

    def forward(self, x, mem):
        if mem is None:
            mem = torch.zeros_like(x)
        return self.lif(x, mem)


class SNNBasicBlock(torch.nn.Module):
    """
    Spiking Residual Block - jeden krok czasowy, stan przekazywany jawnie.
    """
    def __init__(self, in_channels, out_channels, downsampling_layer=None, stride=1, beta=0.9):
        super().__init__()

        self.conv1 = torch.nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride, bias=False)
        self.bn1 = torch.nn.BatchNorm2d(out_channels)
        self.lif1 = StatelessLIF(beta=beta, spike_grad=spike_grad)

        self.conv2 = torch.nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, stride=1, bias=False)
        self.bn2 = torch.nn.BatchNorm2d(out_channels)
        self.lif2 = StatelessLIF(beta=beta, spike_grad=spike_grad)

        self.downsampling_layer = downsampling_layer

    def forward(self, x, mem1, mem2):
        """
        x: (B, C, H, W) - pojedynczy krok czasowy
        zwraca: (spike'i, nowy mem1, nowy mem2)
        """
        res = x

        out = self.conv1(x)
        out = self.bn1(out)
        spk1, mem1 = self.lif1(out, mem1)

        out = self.conv2(spk1)
        out = self.bn2(out)

        if self.downsampling_layer is not None:
            res = self.downsampling_layer(res)

        out = out + res
        spk2, mem2 = self.lif2(out, mem2)

        return spk2, mem1, mem2


# Konstrukcja Sieci

In [24]:
class SmallSNNResNet(torch.nn.Module):
    """
    Spiking ResNet. Stan neuronow (potencjaly blonowe) jest przekazywany jawnie
    jako plaska lista tensorow, co pozwala bezpiecznie checkpointowac kazdy krok czasowy.
    """
    def __init__(self, num_layers, img_channels=2, num_classes=101, beta=0.9,
                 use_final=True, use_checkpoint=True):
        super().__init__()
        self.expansion = 1
        self.use_final = use_final
        self.use_checkpoint = use_checkpoint
        self.in_channels = 64

        # PAMIEC (najwieksza pojedyncza zmiana): stride=2 zamiast stride=1.
        # Standardowy ResNet-18 ma tu stride=2. Przy stride=1 conv1/bn1/lif1
        # trzymaly mape 64 x 128 x 128, a cala reszta sieci pracowala w 4x
        # wiekszej rozdzielczosci niz powinna.
        self.conv1 = torch.nn.Conv2d(img_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = torch.nn.BatchNorm2d(64)
        self.lif1 = StatelessLIF(beta=beta, spike_grad=spike_grad)
        self.maxpool = torch.nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(SNNBasicBlock, num_layers[0], stride=1, out_channels=64, beta=beta)
        self.layer2 = self._make_layer(SNNBasicBlock, num_layers[1], stride=2, out_channels=128, beta=beta)
        self.layer3 = self._make_layer(SNNBasicBlock, num_layers[2], stride=2, out_channels=256, beta=beta)
        self.layer4 = self._make_layer(SNNBasicBlock, num_layers[3], stride=2, out_channels=512, beta=beta)

        self.avg_pool = torch.nn.AdaptiveAvgPool2d((1, 1))

        if use_final:
            self.fc = torch.nn.Linear(512 * self.expansion, num_classes)
            self.lif_out = StatelessLIF(beta=beta, spike_grad=spike_grad)

        # 1 (stem) + 2 na kazdy blok rezydualny + 1 (wyjscie)
        self.n_states = 1 + 2 * sum(num_layers) + (1 if use_final else 0)

    def _make_layer(self, block, num_layers, stride, out_channels, beta):
        layers = []
        downsampling_layer = None
        if stride != 1 or self.in_channels != out_channels * self.expansion:
            downsampling_layer = torch.nn.Sequential(
                torch.nn.Conv2d(self.in_channels, out_channels * self.expansion, kernel_size=1, stride=stride, bias=False),
                torch.nn.BatchNorm2d(out_channels * self.expansion)
            )

        layers.append(block(self.in_channels, out_channels, downsampling_layer, stride, beta=beta))
        self.in_channels = out_channels * self.expansion

        for _ in range(num_layers - 1):
            layers.append(block(self.in_channels, out_channels, beta=beta))

        # ModuleList, nie Sequential - bloki przyjmuja teraz wiecej niz jeden argument
        return torch.nn.ModuleList(layers)

    def forward_single_step(self, x, *mems):
        """
        Jeden krok czasowy. Przyjmuje i zwraca plaska liste stanow,
        zeby dalo sie to opakowac w torch.utils.checkpoint.
        """
        mems = list(mems)
        i = 0

        out = self.conv1(x)
        out = self.bn1(out)
        spk, mems[i] = self.lif1(out, mems[i]); i += 1
        out = self.maxpool(spk)

        for layer in (self.layer1, self.layer2, self.layer3, self.layer4):
            for block in layer:
                out, mems[i], mems[i + 1] = block(out, mems[i], mems[i + 1])
                i += 2

        out = self.avg_pool(out)
        out = out.flatten(1)

        if self.use_final:
            out = self.fc(out)
            spk_out, mems[i] = self.lif_out(out, mems[i]); i += 1
            return (spk_out, *mems)

        return (out, *mems)

    def forward(self, x_seq):
        """
        x_seq: (T, B, C, H, W) lub (B, T, C, H, W)
        """
        if x_seq.dim() == 5 and x_seq.shape[0] != x_seq.shape[1]:
            # zakladamy (B, T, C, H, W) -> (T, B, C, H, W)
            x_seq = x_seq.transpose(0, 1)

        mems = [None] * self.n_states
        spk_rec = []
        mem_rec = []

        for t in range(x_seq.shape[0]):
            if self.use_checkpoint and self.training and torch.is_grad_enabled():
                outs = checkpoint(self.forward_single_step, x_seq[t], *mems, use_reentrant=False)
            else:
                outs = self.forward_single_step(x_seq[t], *mems)

            spk_out = outs[0]
            mems = list(outs[1:])

            spk_rec.append(spk_out)
            mem_rec.append(mems[-1])   # potencjal blonowy warstwy wyjsciowej

        return torch.stack(spk_rec, dim=0), torch.stack(mem_rec, dim=0)


def SNNResNet18(img_channels=2, num_classes=101, device='cuda', beta=0.9, use_checkpoint=True):
    model = SmallSNNResNet(
        num_layers=[2, 2, 2, 2],
        img_channels=img_channels,
        num_classes=num_classes,
        beta=beta,
        use_checkpoint=use_checkpoint,
    ).to(device=device)
    return model


model = SNNResNet18(device=device, use_checkpoint=CONFIG["use_checkpoint"])

n_params = sum(p.numel() for p in model.parameters())
print(f"parametry: {n_params/1e6:.2f} M")


parametry: 11.23 M


In [25]:
dataset_root_path = os.path.join('./data', 'NCALTECH101', 'Caltech101')

# Wykluczamy klase tla - patrz komentarz w CONFIG. Nazwa folderu potwierdzona
# w dystrybucji tonica; jesli assert nizej wywali sie, wypisz
print(os.listdir(dataset_root_path) )
# i popraw nazwe.
EXCLUDED_CLASSES = {"BACKGROUND_Google"}

class_names = sorted([
    d for d in os.listdir(dataset_root_path)
    if os.path.isdir(os.path.join(dataset_root_path, d)) and d not in EXCLUDED_CLASSES
])
class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}

print("liczba klas N-Caltech101 (bez tla):", len(class_names))
assert len(class_names) == CONFIG["num_classes"], (
    "Liczba klas nie zgadza sie z CONFIG['num_classes'] - sprawdz nazwe "
    "folderu klasy tla albo obecnosc klasy 'Faces' w tym zbiorze."
)


def decode_targets(raw_targets):
    idx = [class_to_idx[t.decode('utf-8') if isinstance(t, bytes) else t] for t in raw_targets]
    return torch.tensor(idx, dtype=torch.long, device=device)


# --- sanity check ---------------------------------------------------------
# WAZNE: poprzednio ta petla robila forward BEZ no_grad i zostawiala `loss`,
# `output_spikes` i `data` w zmiennych globalnych. Caly graf obliczeniowy
# zostawal w pamieci GPU przez caly trening. Teraz jest pod no_grad i czyszczony.
model.eval()
with torch.no_grad():
    data, raw_targets = next(iter(train_loader))
    data = data.to(device, non_blocking=True)
    targets = decode_targets(raw_targets)
    output_spikes, output_mem = model(data)
    print("spk_rec:", tuple(output_spikes.shape), "| targets:", tuple(targets.shape))

del data, targets, output_spikes, output_mem, raw_targets
torch.cuda.empty_cache()


['gerenuk', 'tick', 'flamingo', 'wheelchair', 'nautilus', 'laptop', 'octopus', 'inline_skate', 'platypus', 'revolver', 'emu', 'mayfly', 'airplanes', 'Motorbikes', 'cellphone', 'water_lilly', 'watch', 'electric_guitar', 'elephant', 'umbrella', 'dolphin', 'brain', 'lobster', 'crocodile', 'bass', 'hedgehog', 'sea_horse', 'accordion', 'stop_sign', 'cougar_body', 'Faces_easy', 'crayfish', 'headphone', 'rhino', 'minaret', 'snoopy', 'pyramid', 'scissors', 'wrench', 'schooner', 'windsor_chair', 'buddha', 'llama', 'gramophone', 'butterfly', 'sunflower', 'cup', 'dragonfly', 'rooster', 'beaver', 'ibis', 'Leopards', 'ant', 'soccer_ball', 'garfield', 'pizza', 'ewer', 'chandelier', 'scorpion', 'camera', 'okapi', 'cougar_face', 'menorah', 'wild_cat', 'anchor', 'crocodile_head', 'panda', 'ceiling_fan', 'starfish', 'car_side', 'chair', 'grand_piano', 'yin_yang', 'strawberry', 'stegosaurus', 'flamingo_head', 'barrel', 'lamp', 'joshua_tree', 'pigeon', 'binocular', 'bonsai', 'dollar_bill', 'pagoda', 'bron

# Petla Treningowa

In [26]:
import time

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], betas=(0.9, 0.999))
loss_fn = snntorch_functional.mse_count_loss(correct_rate=0.8, incorrect_rate=0.2)

scaler = torch.amp.GradScaler('cuda', enabled=CONFIG["use_amp"])
accum_steps = CONFIG["accum_steps"]

loss_hist = []
train_loss_hist = []
test_loss_hist = []
test_acc_hist = []

torch.cuda.reset_peak_memory_stats()
t_start = time.time()

for epoch in range(CONFIG["epochs"]):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    n_batches = 0

    for i, (data, raw_targets) in enumerate(train_loader):
        if CONFIG["max_steps_per_epoch"] is not None and i >= CONFIG["max_steps_per_epoch"]:
            break

        data = data.to(device, non_blocking=True)
        targets = decode_targets(raw_targets)

        with torch.amp.autocast('cuda', enabled=CONFIG["use_amp"]):
            spk_rec, mem_rec = model(data)
            loss_val = loss_fn(spk_rec, targets)

        # akumulacja gradientu: dzielimy strate, zeby efektywny batch byl
        # batch_size * accum_steps, a pamiec zostala na poziomie batch_size
        scaler.scale(loss_val / accum_steps).backward()

        if (i + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss_val.item()
        loss_hist.append(loss_val.item())
        n_batches += 1

        # jawne zwalnianie duzych tensorow przed nastepna iteracja
        del data, targets, spk_rec, mem_rec, loss_val

        if i % 25 == 0:
            print(f"Epoch {epoch}, Iteration {i} | Train Loss: {loss_hist[-1]:.3f} "
                  f"| peak GPU: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")

    # domkniecie ostatniej, niepelnej akumulacji
    if n_batches % accum_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    train_loss_hist.append(running_loss / max(n_batches, 1))

    # --- ewaluacja --------------------------------------------------------
    model.eval()
    test_loss_total = 0.0
    test_acc_total = 0.0

    with torch.no_grad():
        for data, raw_targets in test_loader:
            data = data.to(device, non_blocking=True)
            targets = decode_targets(raw_targets)

            with torch.amp.autocast('cuda', enabled=CONFIG["use_amp"]):
                spk_rec, mem_rec = model(data)
                loss_val = loss_fn(spk_rec, targets)

            test_loss_total += loss_val.item()
            test_acc_total += snntorch_functional.accuracy_rate(spk_rec, targets)

            del data, targets, spk_rec, mem_rec, loss_val

    avg_test_loss = test_loss_total / len(test_loader)
    avg_test_acc = test_acc_total / len(test_loader)

    test_loss_hist.append(avg_test_loss)
    test_acc_hist.append(avg_test_acc)

    print(f"--- Epoch {epoch} | Train Loss: {train_loss_hist[-1]:.3f} "
          f"| Test Loss: {avg_test_loss:.3f} | Test Acc: {avg_test_acc:.3f}")

elapsed = time.time() - t_start
print(f"\nCzas treningu: {elapsed/60:.1f} min")
print(f"Szczytowe zuzycie pamieci GPU: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")


Epoch 0, Iteration 0 | Train Loss: 0.402 | peak GPU: 14.14 GiB
--- Epoch 0 | Train Loss: 0.372 | Test Loss: 0.213 | Test Acc: 0.005
Epoch 1, Iteration 0 | Train Loss: 0.212 | peak GPU: 14.14 GiB
--- Epoch 1 | Train Loss: 0.157 | Test Loss: 0.128 | Test Acc: 0.010
Epoch 2, Iteration 0 | Train Loss: 0.137 | peak GPU: 14.14 GiB
--- Epoch 2 | Train Loss: 0.106 | Test Loss: 0.125 | Test Acc: 0.008
Epoch 3, Iteration 0 | Train Loss: 0.122 | peak GPU: 14.14 GiB
--- Epoch 3 | Train Loss: 0.098 | Test Loss: 0.108 | Test Acc: 0.095
Epoch 4, Iteration 0 | Train Loss: 0.109 | peak GPU: 14.14 GiB
--- Epoch 4 | Train Loss: 0.095 | Test Loss: 0.111 | Test Acc: 0.004

Czas treningu: 3.2 min
Szczytowe zuzycie pamieci GPU: 14.14 GiB
